# Home Assignment 2
Due by 20th June, 2025 at 23:59 CEST

Students:
* Hauke Schüle, 10004972
* Jannik Jahn, 10020271

### Imports


In [94]:
import pandas as pd
import numpy as np
import torch.nn as nn
import torch
import os
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import gensim.downloader as api
import math
from sklearn.metrics import classification_report

In [95]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


## 1 CNN Classification

In [96]:
df = pd.read_csv('emotion.csv')

In [97]:
label_map = {}
labels = df['label'].unique()
label_names = df['label_name'].unique()
for i, label in enumerate(labels):
    label_map[label] = label_names[i]
print(label_map)

{0: 'sadness', 3: 'anger', 2: 'love', 5: 'surprise', 4: 'fear', 1: 'joy'}


In [98]:
glove = api.load('glove-wiki-gigaword-300')

In [99]:
EMBEDDING_LENGTH = len(glove[0])

In [100]:
df_average = pd.read_csv('emotion.csv')
df_average['text'] = df_average['text'].apply(lambda x: x.lower().split())
sum_lengths = df_average['text'].apply(len).sum()
print(f"Total number of words in the dataset: {sum_lengths}")
total = len(df_average['text'])
print(f"Average number of words per text: {sum_lengths / total:.2f}")
SEQUENCE_LENGTH = math.ceil(sum_lengths / total)
print(f"Average number of words per text rounded up: {SEQUENCE_LENGTH}")

Total number of words in the dataset: 306661
Average number of words per text: 19.17
Average number of words per text rounded up: 20


In [101]:
def collate_fn(batch):
    """
    Custom collate function for processing text data in batches.
    
    Args:
        batch: List of (text, label) tuples
    
    Returns:
        embeddings_tensor: Tensor of shape (batch_size, max_seq_len, embedding_dim)
        labels_tensor: Tensor of labels
    """
    texts, labels = zip(*batch)
    
    # Convert labels to tensor
    labels_tensor = torch.tensor(labels, dtype=torch.long)
    
    # Initialize tensor to hold embeddings for the batch
    batch_embeddings = []
    
    for text in texts:
        # Tokenize text (split into words)
        tokens = text.lower().split()
        
        # Limit sequence length (truncate or pad)
        if len(tokens) > SEQUENCE_LENGTH:
            tokens = tokens[:SEQUENCE_LENGTH]
        
        # Get embeddings for each word
        embeddings = []
        for token in tokens:
            if token in glove:
                embeddings.append(glove[token])
            else:
                embeddings.append(np.zeros(EMBEDDING_LENGTH))
                
        # Pad sequences shorter than max_seq_len
        padding_length = SEQUENCE_LENGTH - len(tokens)
        for _ in range(padding_length):
            embeddings.append(np.zeros(EMBEDDING_LENGTH))
            
        # Convert to tensor and add to batch
        batch_embeddings.append(torch.tensor(embeddings, dtype=torch.float))
    
    # Stack all embeddings in batch
    embeddings_tensor = torch.stack(batch_embeddings)
    
    return embeddings_tensor, labels_tensor

In [102]:
class EmotionDataset(Dataset):
    def __init__(self, df):
        self.df = df
        self.labels = df['label'].values
        self.texts = df['text'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        return text, label

In [103]:
emotion_dataset = EmotionDataset(df)
train_set, val_set, test_set = torch.utils.data.random_split(emotion_dataset, [int(len(emotion_dataset) * 0.8), int(len(emotion_dataset) * 0.1), int(len(emotion_dataset) * 0.1)])

In [104]:
train_loader = DataLoader(train_set, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False, collate_fn=collate_fn)

In [105]:
for batch in train_loader:
    embeddings, labels = batch
    print(f"Embeddings shape: {embeddings.shape}, Labels shape: {labels.shape}")
    break

Embeddings shape: torch.Size([32, 20, 300]), Labels shape: torch.Size([32])


In [106]:
# HYPERPARAMETERS

KERNEL_SIZE = 2
HIDDEN_DIM = 128
NUM_CLASSES = len(label_map)
DROPOUT = 0.5
LEARNING_RATE = 0.001
EPOCHS = 10

In [107]:
def validate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    
    with torch.no_grad():
        for embeddings, labels in tqdm(val_loader, desc="Validation"):
            embeddings = embeddings.to(device)
            labels = labels.to(device)
            
            outputs = model(embeddings)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
    
    validation_loss = total_loss / len(val_loader)
    return validation_loss

In [108]:
def train(model, train_loader, val_loader, criterion, optimizer, epochs, device):
    model.to(device)
    min_validation_loss = float('inf')
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        train_loader = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}")
        for i, (embeddings, labels) in enumerate(train_loader):
            embeddings = embeddings.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(embeddings)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()

            train_loader.set_postfix({"loss": total_loss / (i + 1)})
        validation_loss = validate(model, val_loader, criterion, device)
        if validation_loss < min_validation_loss:
            min_validation_loss = validation_loss
            torch.save(model.state_dict(), 'best_model.pth')
            print(f"Validation loss improved to {min_validation_loss:.4f}, model saved.")
    
    return total_loss / len(train_loader)

In [109]:
# Model architecture

class CNN(nn.Module):
    def __init__(self, num_classes, dropout=0.5):
        super(CNN, self).__init__()
        self.conv = nn.Conv2d(in_channels=1, out_channels=HIDDEN_DIM, kernel_size=(KERNEL_SIZE, EMBEDDING_LENGTH))
        self.maxpool = nn.MaxPool1d(kernel_size=SEQUENCE_LENGTH - KERNEL_SIZE + 1)
        self.dropout = nn.Dropout(dropout)
        self.linear = nn.Linear(HIDDEN_DIM, num_classes)

    def forward(self, x):
        x = self.conv(x.unsqueeze(1))
        x = x.squeeze(-1)
        x = self.maxpool(x)
        x = x.squeeze(-1)
        x = self.dropout(x)
        x = self.linear(x)
        return x

In [110]:
# Model instantiation
model = CNN(num_classes=NUM_CLASSES, dropout=DROPOUT).to(DEVICE)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)


In [111]:
train(
    model,
    train_loader,
    val_loader,
    criterion, 
    optimizer, 
    EPOCHS, 
    DEVICE
)

Validation: 100%|██████████| 50/50 [00:00<00:00, 88.30it/s]


Validation loss improved to 0.6974, model saved.


Validation: 100%|██████████| 50/50 [00:00<00:00, 86.95it/s]


Validation loss improved to 0.5557, model saved.


Validation: 100%|██████████| 50/50 [00:00<00:00, 90.83it/s]


Validation loss improved to 0.5154, model saved.


Validation: 100%|██████████| 50/50 [00:00<00:00, 91.16it/s]


Validation loss improved to 0.5067, model saved.


Validation: 100%|██████████| 50/50 [00:00<00:00, 88.89it/s]


0.3111158131249249

In [112]:
model.load_state_dict(torch.load('best_model.pth'))


<All keys matched successfully>

In [120]:
def evaluate(model, test_loader):
    model.to('cpu')
    model.eval()
    all_labels = []
    all_preds = []
    
    with torch.no_grad():
        for embeddings, labels in tqdm(test_loader, desc="Testing"):
            embeddings = embeddings
            labels = labels
            
            outputs = model(embeddings)
            _, preds = torch.max(outputs, 1)
            
            all_labels.extend(labels.tolist())
            all_preds.extend(preds.tolist())
    
    return all_labels, all_preds

In [122]:
y_true, y_pred = evaluate(model, test_loader)
print(classification_report(y_true, y_pred, target_names=label_names))

Testing: 100%|██████████| 50/50 [00:00<00:00, 70.60it/s]

              precision    recall  f1-score   support

     sadness       0.86      0.86      0.86       482
       anger       0.81      0.87      0.84       515
        love       0.77      0.70      0.73       130
    surprise       0.81      0.84      0.82       208
        fear       0.85      0.75      0.80       211
         joy       0.69      0.67      0.68        54

    accuracy                           0.82      1600
   macro avg       0.80      0.78      0.79      1600
weighted avg       0.82      0.82      0.82      1600

